[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/A.%20Foundations%20of%20Terminal%20Operations%20%28The%20Building%20Blocks%29/01.%20The%20Single%20Crane%20Lift%20Sequence%20Problem/P1-Tier-2_executed.ipynb)

---


# 1. The Single Crane Lift Sequence Problem

## Tier 2 — A WSPT-Inspired Greedy Heuristic

This notebook builds a complete lift sequence for a small container bay. At each step, a transparent greedy rule compares the most attractive feasible single lift with the most attractive feasible dual lift. The rule uses container priority and estimated time, with a threshold controlling how readily it accepts a dual lift.

The method is inspired by **Weighted Shortest Processing Time (WSPT)**: in the classical setting, a job's weight represents its importance and its processing time represents the time it occupies a resource. Here, travel, handling, and mode-switching times make the candidate costs depend on the crane's current state. The resulting rule is therefore a heuristic, not an exact WSPT schedule and not a guarantee of a globally optimal sequence.

### Learning goals

- Define and interpret a priority-per-time heuristic score.
- Track the crane's location and operating mode as the sequence is constructed.
- Check dual-lift feasibility using adjacency and a weight limit.
- Understand how a threshold affects the greedy choice.
- Read the resulting schedule and relate its time components to visualizations.

### Inputs and outputs

The input is an 8-container instance on a 2 × 4 grid, together with handling times, a setup time, a dual-lift weight limit, a travel-time rate, and the crane's initial location and mode. The output is a step-by-step schedule, its total completion time, sensitivity tables, and visual summaries of the schedule.

### Scope and modeling assumptions

Travel time is proportional to Manhattan distance on the grid. A single lift visits one container's grid cell; a dual lift visits the midpoint of two eligible adjacent containers. A mode change incurs a fixed setup time before handling. Handling times are fixed, and all containers must be lifted. The model does not represent blocking, crane interference, uncertainty, deadlines, or other operational constraints. Priority is an urgency weight in the heuristic score; it is not a promised completion time or a hard scheduling constraint.


In [ ]:
# Check that the notebook's existing scientific Python dependencies are available.
# Dependencies are expected to be installed in the notebook environment.
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError as error:
    raise ImportError(
        "Missing dependency. Install numpy, pandas, and matplotlib in the "
        "notebook environment before running this notebook."
    ) from error

print("Dependencies imported successfully.")


## Concrete example instance (8 containers)

The bay is a 2 × 4 grid. Each container has a unique identifier, a grid location, a weight in tons, and an integer priority. A larger priority means greater urgency in the heuristic.

### Time and geometry model

For locations \(a=(r_a,c_a)\) and \(b=(r_b,c_b)\), Manhattan distance is

$$d_1(a,b)=|r_a-r_b|+|c_a-c_b|.$$

Travel time is the distance multiplied by a constant number of minutes per grid step. A single lift travels to one container's cell and handles that container. A dual lift travels to the midpoint of two containers and handles both at once. The crane has a mode, either `single` or `dual`; changing modes adds the fixed setup time to that step.

### Dual-lift feasibility

A pair is eligible for a dual lift only when its containers are orthogonally adjacent—one Manhattan grid step apart—and their combined weight does not exceed the dual weight limit. Diagonal neighbors are not adjacent under this definition. The midpoint is the coordinate-wise average of the two container locations and can therefore have half-integer coordinates.


In [ ]:
# Define the container instance, operating parameters, and geometry helpers.
from dataclasses import dataclass
from itertools import combinations
from typing import Any, Dict, List, Literal, Optional, Set, Tuple
import ast
import math

Mode = Literal["single", "dual"]


@dataclass(frozen=True)
class Container:
    """A container with a grid location, weight in tons, and priority."""

    id: int
    rc: Tuple[int, int]
    weight: float
    priority: int


# The fixed instance: coordinates are (row, column).
containers: List[Container] = [
    Container(1, (0, 0), 8.5, 3),
    Container(2, (0, 1), 12.3, 2),
    Container(3, (0, 2), 15.7, 4),
    Container(4, (0, 3), 9.8, 3),
    Container(5, (1, 0), 11.2, 2),
    Container(6, (1, 1), 14.1, 2),
    Container(7, (1, 2), 13.9, 1),
    Container(8, (1, 3), 10.6, 2),
]

id_to_container = {container.id: container for container in containers}
assert len(id_to_container) == len(containers), "Container IDs must be unique."

# Time parameters are measured in minutes unless stated otherwise.
HANDLE_SINGLE = 2.5
HANDLE_DUAL = 3.8
SETUP_TIME = 1.2
DUAL_WEIGHT_LIMIT = 20.0
TRAVEL_PER_GRID_STEP = 0.8

# Initial crane state.
START_RC = (0, 0)
START_MODE: Mode = "single"


def manhattan(
    a: Tuple[float, float], b: Tuple[float, float]
) -> float:
    """Return Manhattan distance between two grid coordinates."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def travel_time(
    a: Tuple[float, float],
    b: Tuple[float, float],
    travel_per_grid_step: float,
) -> float:
    """Convert Manhattan distance into travel time in minutes."""
    return travel_per_grid_step * manhattan(a, b)


def adjacent_int(a: Tuple[int, int], b: Tuple[int, int]) -> bool:
    """Return whether integer grid cells share an orthogonal edge."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1]) == 1


def dual_feasible(
    i: int,
    j: int,
    id_to_container: Dict[int, Container],
    dual_weight_limit: float,
) -> bool:
    """Check adjacency and capacity for a candidate dual lift."""
    container_i = id_to_container[i]
    container_j = id_to_container[j]
    adjacent = adjacent_int(container_i.rc, container_j.rc)
    within_capacity = (
        container_i.weight + container_j.weight <= dual_weight_limit
    )
    return adjacent and within_capacity


def dual_center(
    container_i: Container, container_j: Container
) -> Tuple[float, float]:
    """Return the midpoint target for a dual lift."""
    return (
        (container_i.rc[0] + container_j.rc[0]) / 2.0,
        (container_i.rc[1] + container_j.rc[1]) / 2.0,
    )


# This list describes eligibility under the instance's stated weight limit.
# The heuristic checks feasibility again using its supplied limit so that
# sensitivity experiments remain consistent with their parameter values.
dual_pairs = [
    (i, j)
    for i, j in combinations(sorted(id_to_container), 2)
    if dual_feasible(i, j, id_to_container, DUAL_WEIGHT_LIMIT)
]

dual_pairs


## Heuristic logic (greedy decision at each step)

At each step, the heuristic evaluates the best available single lift and the best feasible dual lift, then makes a local choice. “Greedy” means that this choice uses the current crane state and the candidates available now; it does not search all possible future sequences.

### Candidate scores

A higher-priority lift is attractive, while a lift with a longer estimated time is less attractive. For a remaining container \(i\), define its single-lift score as

$$S_i=\frac{p_i}{t_i^{\mathrm{travel}}+s_i^{\mathrm{setup}}+h^{\mathrm{single}}+\varepsilon}.$$

For a feasible pair \((i,j)\), define the dual-lift score as

$$D_{ij}=\frac{(p_i+p_j)/2}{t_{ij}^{\mathrm{travel}}+s_{ij}^{\mathrm{setup}}+h^{\mathrm{dual}}+\varepsilon}.$$

Here, \(p_i\) is container \(i\)'s priority; \(t^{\mathrm{travel}}\) is travel time from the current crane location to the relevant cell or midpoint; \(s^{\mathrm{setup}}\) is the mode-switching time, or zero if the mode does not change; \(h^{\mathrm{single}}\) and \(h^{\mathrm{dual}}\) are the corresponding handling times; and \(\varepsilon=10^{-9}\) prevents division by zero. The dual score uses average priority as a simple proxy for the pair's urgency. These scores are decision indices, not objective values or probabilities.

### Threshold decision

Let \(S^*\) be the largest single score and \(D^*\) the largest feasible dual score. If at least one feasible pair exists and

$$D^*\geq \theta S^*,$$

the heuristic selects the best dual pair; otherwise it selects the best single container. The parameter \(\theta\) is the threshold. A larger threshold requires a dual lift to be more competitive; a threshold of zero selects a dual lift whenever one is feasible. The rule is repeated after updating the crane location, mode, and set of remaining containers.

### Computational effort

With \(n\) containers, a single iteration evaluates at most \(n\) single candidates and \(\binom{n}{2}\) pairs. If one lift is performed per iteration, there are at most \(n\) iterations, so the direct implementation has worst-case time complexity \(O(n^3)\) and uses \(O(n)\) schedule storage, aside from the instance data. A dual lift can remove two containers in one iteration. The method is useful as an interpretable heuristic, but it does not certify that the resulting total time is globally minimal.


In [ ]:
# Implement candidate scoring, the greedy schedule, and parameter sensitivity.
def best_single(
    remaining: Set[int],
    crane_rc: Tuple[float, float],
    crane_mode: Mode,
    id_to_container: Dict[int, Container],
    handle_single: float,
    setup_time: float,
    travel_per_grid_step: float,
) -> Tuple[int, float]:
    """Return the best single-lift container ID and its heuristic score."""
    eps = 1e-9
    best_id: Optional[int] = None
    best_score = -1.0

    # Sorting makes equal-score choices reproducible by container ID.
    for container_id in sorted(remaining):
        container = id_to_container[container_id]
        target = (float(container.rc[0]), float(container.rc[1]))
        travel = travel_time(crane_rc, target, travel_per_grid_step)
        setup = setup_time if crane_mode != "single" else 0.0
        time_cost = travel + setup + handle_single
        score = container.priority / (time_cost + eps)

        # Keep the first candidate when scores are equal within tolerance.
        tied = math.isclose(
            score, best_score, rel_tol=1e-12, abs_tol=1e-12
        )
        if score > best_score and not tied:
            best_score = score
            best_id = container_id

    assert best_id is not None
    return best_id, float(best_score)


def best_dual(
    remaining: Set[int],
    crane_rc: Tuple[float, float],
    crane_mode: Mode,
    id_to_container: Dict[int, Container],
    dual_weight_limit: float,
    setup_time: float,
    handle_dual: float,
    travel_per_grid_step: float,
) -> Tuple[Optional[Tuple[int, int]], float]:
    """Return the best feasible pair and its score, or (None, -1)."""
    eps = 1e-9
    best_pair: Optional[Tuple[int, int]] = None
    best_score = -1.0

    for i, j in combinations(sorted(remaining), 2):
        if not dual_feasible(i, j, id_to_container, dual_weight_limit):
            continue

        container_i = id_to_container[i]
        container_j = id_to_container[j]
        target = dual_center(container_i, container_j)
        travel = travel_time(crane_rc, target, travel_per_grid_step)
        setup = setup_time if crane_mode != "dual" else 0.0
        average_priority = (
            container_i.priority + container_j.priority
        ) / 2.0
        time_cost = travel + setup + handle_dual
        score = average_priority / (time_cost + eps)

        # Sorted pair enumeration provides a deterministic tie rule.
        tied = math.isclose(
            score, best_score, rel_tol=1e-12, abs_tol=1e-12
        )
        if score > best_score and not tied:
            best_score = score
            best_pair = (i, j)

    return best_pair, float(best_score)


def run_heuristic(
    threshold: float = 0.9,
    container_data: List[Container] = containers,
    start_rc: Tuple[int, int] = START_RC,
    start_mode: Mode = START_MODE,
    handle_single: float = HANDLE_SINGLE,
    handle_dual: float = HANDLE_DUAL,
    setup_time: float = SETUP_TIME,
    dual_weight_limit: float = DUAL_WEIGHT_LIMIT,
    travel_per_grid_step: float = TRAVEL_PER_GRID_STEP,
) -> Tuple[pd.DataFrame, float]:
    """Build a greedy schedule and return its step table and total time."""
    if threshold < 0:
        raise ValueError("The threshold must be nonnegative.")

    id_to_container = {container.id: container for container in container_data}
    if len(id_to_container) != len(container_data):
        raise ValueError("Container IDs must be unique.")

    remaining = set(id_to_container)
    crane_rc: Tuple[float, float] = (
        float(start_rc[0]),
        float(start_rc[1]),
    )
    crane_mode = start_mode
    steps: List[Dict[str, Any]] = []
    total_time = 0.0

    while remaining:
        single_id, single_score = best_single(
            remaining,
            crane_rc,
            crane_mode,
            id_to_container,
            handle_single,
            setup_time,
            travel_per_grid_step,
        )
        dual_pair, dual_score = best_dual(
            remaining,
            crane_rc,
            crane_mode,
            id_to_container,
            dual_weight_limit,
            setup_time,
            handle_dual,
            travel_per_grid_step,
        )

        boundary_score = threshold * single_score
        dual_is_competitive = (
            dual_score >= boundary_score
            or math.isclose(
                dual_score,
                boundary_score,
                rel_tol=1e-12,
                abs_tol=1e-12,
            )
        )
        choose_dual = dual_pair is not None and dual_is_competitive

        if choose_dual:
            i, j = dual_pair
            container_i = id_to_container[i]
            container_j = id_to_container[j]
            target = dual_center(container_i, container_j)
            travel = travel_time(
                crane_rc, target, travel_per_grid_step
            )
            setup = setup_time if crane_mode != "dual" else 0.0
            step_time = travel + setup + handle_dual

            steps.append(
                {
                    "type": "dual",
                    "containers": f"{i},{j}",
                    "to": f"center({target[0]:.1f},{target[1]:.1f})",
                    "travel": travel,
                    "setup": setup,
                    "handle": handle_dual,
                    "step_time": step_time,
                }
            )

            total_time += step_time
            remaining.difference_update({i, j})
            crane_rc = target
            crane_mode = "dual"
        else:
            container = id_to_container[single_id]
            target = (
                float(container.rc[0]),
                float(container.rc[1]),
            )
            travel = travel_time(
                crane_rc, target, travel_per_grid_step
            )
            setup = setup_time if crane_mode != "single" else 0.0
            step_time = travel + setup + handle_single

            steps.append(
                {
                    "type": "single",
                    "containers": f"{single_id}",
                    "to": str(container.rc),
                    "travel": travel,
                    "setup": setup,
                    "handle": handle_single,
                    "step_time": step_time,
                }
            )

            total_time += step_time
            remaining.remove(single_id)
            crane_rc = target
            crane_mode = "single"

    step_table = pd.DataFrame(steps)
    step_table.index = np.arange(1, len(step_table) + 1)

    numeric_columns = ["travel", "setup", "handle", "step_time"]
    step_table[numeric_columns] = step_table[numeric_columns].astype(float)

    # These checks ensure that the schedule is complete and its time total
    # agrees with the sum of the travel, setup, and handling components.
    scheduled_ids = []
    for scheduled_step in steps:
        scheduled_ids.extend(
            int(value)
            for value in scheduled_step["containers"].split(",")
        )
    assert sorted(scheduled_ids) == sorted(id_to_container)
    assert math.isclose(
        total_time,
        sum(step["step_time"] for step in steps),
        rel_tol=1e-12,
        abs_tol=1e-12,
    )
    for step in steps:
        assert math.isclose(
            step["step_time"],
            step["travel"] + step["setup"] + step["handle"],
            rel_tol=1e-12,
            abs_tol=1e-12,
        )

    return step_table, float(total_time)


# Run the specified instance using the default threshold.
df_steps, total_time = run_heuristic(threshold=0.9)
print(f"Total heuristic time: {total_time:.3f} min")
num_dual = int((df_steps["type"] == "dual").sum())
print(f"Dual steps used (threshold=0.9): {num_dual}")
df_steps


# Vary the decision threshold while keeping the instance and weight limit fixed.
thresholds = [0.95, 0.9, 0.85, 0.8, 0.7, 0.0]
threshold_rows = []
for threshold_value in thresholds:
    threshold_schedule, threshold_total = run_heuristic(
        threshold=threshold_value
    )
    threshold_rows.append(
        {
            "threshold": float(threshold_value),
            "total_time": float(threshold_total),
            "dual_steps": int(
                (threshold_schedule["type"] == "dual").sum()
            ),
        }
    )

print("\nSensitivity over threshold (fixed weight limit):")
pd.DataFrame(threshold_rows)


def run_with_dual_limit(limit: float, threshold: float) -> Dict[str, Any]:
    """Run the heuristic at a specified dual weight limit."""
    limited_schedule, limited_total = run_heuristic(
        threshold=threshold,
        dual_weight_limit=float(limit),
    )
    return {
        "dual_weight_limit": float(limit),
        "threshold": float(threshold),
        "total_time": float(limited_total),
        "dual_steps": int(
            (limited_schedule["type"] == "dual").sum()
        ),
    }


limits = [20.0, 22.0, 24.0, 26.0, 30.0]

print("\nSensitivity over dual weight limit (threshold fixed at 0.85):")
limit_rows = [
    run_with_dual_limit(limit, threshold=0.85) for limit in limits
]
pd.DataFrame(limit_rows)

print(
    "\nSensitivity over dual weight limit "
    "(threshold fixed at 0.0 to select dual when feasible):"
)
forced_dual_rows = [
    run_with_dual_limit(limit, threshold=0.0) for limit in limits
]
pd.DataFrame(forced_dual_rows)


## Visualization: crane path through the bay

The diagram places containers at their grid locations and draws the crane's successive target locations in schedule order. A single-lift target is a container cell; a dual-lift target is the pair's midpoint. Compare the path with the `to` and `type` columns in the schedule table to see which action each target represents.


In [ ]:
# Draw the bay and the sequence of crane target locations.
fig, ax = plt.subplots(figsize=(7, 3.2))

for row in [0, 1]:
    for column in [0, 1, 2, 3]:
        ax.scatter(
            [column],
            [-row],
            s=900,
            marker="s",
            facecolor="#F2F4F7",
            edgecolor="#344054",
        )

for container in containers:
    x, y = container.rc[1], -container.rc[0]
    ax.text(
        x,
        y,
        f"{container.id}\n(w={container.weight}, p={container.priority})",
        ha="center",
        va="center",
        fontsize=9,
    )

# Plot coordinates use column as x and the negative row as y.
path = [(START_RC[1], -START_RC[0])]
for _, schedule_row in df_steps.iterrows():
    if schedule_row["type"] == "single":
        target = ast.literal_eval(schedule_row["to"])
        path.append((target[1], -target[0]))
    else:
        target_text = schedule_row["to"]
        coordinates = target_text[
            target_text.find("(") + 1 : target_text.find(")")
        ]
        target_row, target_column = [
            float(value) for value in coordinates.split(",")
        ]
        path.append((target_column, -target_row))

for (x_start, y_start), (x_end, y_end) in zip(path[:-1], path[1:]):
    ax.annotate(
        "",
        xy=(x_end, y_end),
        xytext=(x_start, y_start),
        arrowprops={"arrowstyle": "->", "lw": 2, "color": "#2E90FA"},
    )

ax.set_title("Greedy heuristic: crane path")
ax.set_xlabel("Grid column")
ax.set_ylabel("Grid row (shown vertically inverted)")
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlim(-0.75, 3.75)
ax.set_ylim(-1.75, 0.75)
ax.set_aspect("equal")
plt.show()


## Operational visualizations

The schedule table can also be read as an operations summary. The following plots show how each step's time is divided among travel, setup, and handling; how completion time accumulates; and when each action occurs on a simple timeline. The charts use the schedule table directly. Values in the table are kept at full precision; displayed text and axes may be rounded for readability.


In [ ]:
# Prepare cumulative times and timeline intervals from the schedule table.
vis = df_steps.copy()
vis["step"] = vis.index.astype(int)
vis["cum_time"] = vis["step_time"].cumsum()
vis["start_time"] = vis["cum_time"] - vis["step_time"]

color_map = {"single": "#2E90FA", "dual": "#12B76A"}
colors = [color_map[action] for action in vis["type"].tolist()]

fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))

# Stacked bars show the modeled time components for each step.
axes[0].bar(
    vis["step"], vis["travel"], label="Travel", color="#98A2B3"
)
axes[0].bar(
    vis["step"],
    vis["setup"],
    bottom=vis["travel"],
    label="Setup",
    color="#F97066",
)
axes[0].bar(
    vis["step"],
    vis["handle"],
    bottom=vis["travel"] + vis["setup"],
    label="Handling",
    color="#FDB022",
)
axes[0].set_title("Time breakdown per step")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Time (minutes)")
axes[0].grid(True, axis="y", alpha=0.25)
axes[0].legend()

# The cumulative curve ends at the total modeled completion time.
axes[1].plot(
    vis["step"],
    vis["cum_time"],
    marker="o",
    linewidth=2,
    color="#344054",
)
axes[1].set_title("Cumulative completion time")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Time (minutes)")
axes[1].grid(True, alpha=0.25)

# Each bar spans [start_time, start_time + step_time) on the timeline.
axes[2].barh(
    y=vis["step"],
    width=vis["step_time"],
    left=vis["start_time"],
    color=colors,
    edgecolor="#101828",
    alpha=0.85,
)
axes[2].set_title("Action timeline")
axes[2].set_xlabel("Time (minutes)")
axes[2].set_ylabel("Step")
axes[2].grid(True, axis="x", alpha=0.25)

from matplotlib.patches import Patch

legend_handles = [
    Patch(
        facecolor=color_map["single"],
        edgecolor="#101828",
        label="Single lift",
    ),
    Patch(
        facecolor=color_map["dual"],
        edgecolor="#101828",
        label="Dual lift",
    ),
]
axes[2].legend(handles=legend_handles, loc="best")

plt.tight_layout()
plt.show()


## Interpreting the heuristic

The schedule is a complete, feasible sequence under the assumptions encoded in this notebook. Its reported total is the sum of travel, setup, and handling time for every selected action. A dual lift may reduce the number of handling actions, but its midpoint travel, handling duration, capacity restriction, and possible mode-switching setup all affect its score and its realized time.

A threshold or weight-limit sensitivity table shows how the specified greedy rule responds when that parameter changes. It does not establish that one setting is best for all instances, and a shorter total for one setting does not prove global optimality. In particular, a dual action may be absent from the default schedule because no feasible pair is sufficiently competitive under the current state and threshold. The sensitivity experiments provide a way to inspect dual-lift behavior without changing the main instance parameters.

### Strengths and limitations

- The score, feasibility checks, and threshold make each local decision inspectable.
- Candidate evaluation is direct and the schedule is constructed one action at a time.
- The rule is not guaranteed to minimize total completion time: a locally attractive action can leave a less favorable crane location or mode for later steps.
- The resulting schedule depends on the priority score, threshold, and modeling assumptions; operational constraints outside this model are not represented.

This heuristic is most useful as a fast, interpretable scheduling rule for exploring candidate sequences and understanding the effect of the model's inputs.
